# Зарплаты в вакансиях Авито

Сначала пробовали оценивать сменные предложения с хорошо описанными условиями. Затем добавили объявления без указанного периода и выделили суммы «от» и «до» в отдельные задачи.

Для повтора нужен исходный `Data/dataset/vacancies.csv` из проекта. Для точного повтора скачайте [CSV на момент эксперимента](https://raw.githubusercontent.com/tooscl/salary-navigator/fdbbaf1124dc675e7b7cd9ca6f9abc9b6f1a47d1/Data/dataset/vacancies.csv). Положите его рядом с ноутбуком под именем `vacancies.csv` и запустите все ячейки. В репозитории ноутбук также ищет CSV в `Data/dataset/`. Для другого расположения укажите путь в `DATA_PATH`. Первая ячейка проверяет SHA-256.

Окружение исходного расчёта: Python 3.10, NumPy 1.26.4, pandas 2.0.1, scikit-learn 1.5.2. Для запуска в Jupyter установите зависимости командой `python -m pip install numpy==1.26.4 pandas==2.0.1 scikit-learn==1.5.2 ipykernel`. Затем выберите это окружение и выполните все ячейки.

In [1]:
import hashlib
import json
import platform
import re
from dataclasses import asdict, dataclass
from pathlib import Path

import numpy as np
import pandas as pd
import sklearn
from IPython.display import display
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import GradientBoostingRegressor, RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, confusion_matrix, f1_score, mean_absolute_error,
    mean_pinball_loss, recall_score,
)
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder

DATA_PATH = None  # Например, Path("vacancies.csv")
ALLOW_DIFFERENT_DATA = False
EXPECTED_SHA256 = "c88bbf056d942b921eaaf4475289a1f0b7d03ef2574957f10c1b60a4f835d508"

if DATA_PATH is None and Path("vacancies.csv").is_file():
    DATA_PATH = Path("vacancies.csv")

if DATA_PATH is None:
    cwd = Path.cwd().resolve()
    DATA_PATH = next(
        (folder / "Data/dataset/vacancies.csv" for folder in (cwd, *cwd.parents)
         if (folder / "Data/dataset/vacancies.csv").is_file()),
        None,
    )
if DATA_PATH is None or not Path(DATA_PATH).is_file():
    raise FileNotFoundError("Укажите путь к исходному vacancies.csv в DATA_PATH.")

DATA_PATH = Path(DATA_PATH)
with DATA_PATH.open("rb") as stream:
    digest = hashlib.sha256()
    for chunk in iter(lambda: stream.read(1024 * 1024), b""):
        digest.update(chunk)
if digest.hexdigest() != EXPECTED_SHA256:
    message = "Хеш выгрузки отличается. Скачайте CSV по ссылке выше или укажите ALLOW_DIFFERENT_DATA = True."
    if not ALLOW_DIFFERENT_DATA:
        raise ValueError(message)
    print(message)

raw = pd.read_csv(DATA_PATH, dtype=str, keep_default_na=False)
print(f"{len(raw):,} объявлений · SHA-256 {digest.hexdigest()}")

17,691 объявлений · SHA-256 c88bbf056d942b921eaaf4475289a1f0b7d03ef2574957f10c1b60a4f835d508


## Что считаем зарплатой

Берём опубликованную сумму или середину закрытой вилки. Если указано только «от» или «до», прогнозируем заявленную границу. Оплату за смену, час, неделю и сдельную оплату разбираем отдельно. Объявления без указанного периода образуют свою группу. Все суммы относятся к объявлениям, фактические выплаты в этих расчётах неизвестны.

In [2]:
def clean(value) -> str:
    if pd.isna(value):
        return ""
    return re.sub(r"\s+", " ", str(value).replace("\\r\\n", " ")
                  .replace("\\n", " ").replace("\\r", " ")).strip()


def parse_salary(value) -> dict:
    """Parse only complete, evidenced RUB expressions; never infer a period."""
    text = clean(value).lower().replace("ё", "е")
    result = dict(salary_parse_status="unparsed", salary_lower=np.nan,
                  salary_upper=np.nan, salary_form="unknown", pay_unit="unknown",
                  tax_basis="unknown", salary_target=np.nan, range_ratio=np.nan)
    if not text or "не указана" in text:
        result["salary_parse_status"] = "missing"
        return result
    unit_patterns = {"shift": r"за смену", "hour": r"(?:в|за) час",
                     "month": r"(?:в|за) месяц", "piece": r"сдельно"}
    units = [u for u, pattern in unit_patterns.items() if re.search(pattern, text)]
    result["pay_unit"] = units[0] if len(units) == 1 else "unknown"
    net = "на руки" in text
    gross = bool(re.search(r"до (?:вычета|уплаты) налогов|до ндфл", text))
    result["tax_basis"] = "conflict" if net and gross else "net" if net else "gross" if gross else "unknown"
    rest = text
    for pattern in unit_patterns.values():
        rest = re.sub(pattern, " ", rest)
    rest = re.sub(r"на руки|до (?:вычета|уплаты) налогов|до ндфл", " ", rest)
    # Require a currency, full match and conventional thousands separators.
    number = r"(?:\d{1,3}(?: \d{3})+|\d+)(?:[.,]\d{1,2})?"
    match = re.fullmatch(rf"\s*(?:(от|до)\s+)?({number})(?:\s*[—–-]\s*({number}))?\s*(?:₽|руб\.?|р\.)\s*", rest)
    if not match or len(units) > 1 or result["tax_basis"] == "conflict":
        return result
    prefix, a, b = match.groups()
    if prefix and b:
        return result
    a = float(a.replace(" ", "").replace(",", "."))
    b = float(b.replace(" ", "").replace(",", ".")) if b else None
    if not np.isfinite(a) or a <= 0 or (b is not None and (not np.isfinite(b) or b < a)):
        result["salary_parse_status"] = "invalid_bounds"
        return result
    result["salary_parse_status"] = "parsed"
    if prefix == "от":
        result.update(salary_form="lower_only", salary_lower=a)
    elif prefix == "до":
        result.update(salary_form="upper_only", salary_upper=a)
    else:
        b = a if b is None else b
        result.update(salary_form="exact" if a == b else "closed_range",
                      salary_lower=a, salary_upper=b, salary_target=a + (b - a) / 2,
                      range_ratio=b / a)
    return result

In [3]:
def parse_offer(value):
    """Extend v1 conservatively; keep old module and outputs reproducible."""
    text = clean(value)
    extras = [(unit, pattern) for unit, pattern in [
        ("week", r"\b(?:в|за) неделю\b"), ("day", r"\b(?:в|за) день\b")]
        if re.search(pattern, text, flags=re.I)]
    if not extras:
        return parse_salary(value)
    stripped = text
    for _, pattern in extras:
        stripped = re.sub(pattern, " ", stripped, flags=re.I)
    parsed = parse_salary(stripped)
    if len(extras) != 1 or parsed["pay_unit"] != "unknown":
        parsed.update(salary_parse_status="unparsed", pay_unit="conflict")
    else:
        parsed["pay_unit"] = extras[0][0]
    return parsed

## Как делим объявления

Вакансии одного работодателя и связанные дубли держим в одной части выборки. Номер разбиения определяет, какие группы попадают в обучение, валидацию и тест. В расширенном эксперименте группы строятся до отбора вариантов обучения.

In [4]:
def assign_groups(frame: pd.DataFrame, seed: int) -> pd.DataFrame:
    """Keep employer identities and identical saved offers in one split.

    Union-find handles an employer with multiple profile URLs/names and offers
    duplicated under other IDs. Names are an imperfect employer proxy.
    """
    parents = list(range(len(frame)))

    def find(i):
        while parents[i] != i:
            parents[i] = parents[parents[i]]
            i = parents[i]
        return i

    seen = {}
    for i, row in enumerate(frame.to_dict("records")):
        tokens = []
        for field in ["employer_name_observed", "employer_profile_url"]:
            value = clean(row.get(field, "")).casefold()
            if value:
                tokens.append(field + ":" + value.split("?")[0])
        # Ignore salary when identifying identical offer text.
        signature = [clean(row.get(k, "")).casefold()
                     for k in ["city", "profession", "title", "description"]]
        if signature[-1]:
            tokens.append("offer:" + hashlib.sha256(json.dumps(signature, ensure_ascii=False).encode()).hexdigest())
        for token in tokens:
            if token in seen:
                parents[find(i)] = find(seen[token])
            seen[token] = i
    keys = frame["source"] + ":" + frame["id"]
    members = {}
    for i, key in enumerate(keys):
        members.setdefault(find(i), []).append(key)
    group_names = {k: min(v) for k, v in members.items()}
    result = frame.copy()
    result["split_group"] = [hashlib.sha256(group_names[find(i)].encode()).hexdigest()[:20]
                             for i in range(len(frame))]
    def split(group):
        bucket = int(hashlib.sha256(f"{seed}:{group}".encode()).hexdigest()[:8], 16) % 100
        return "train" if bucket < 70 else "validation" if bucket < 85 else "test"
    result["split"] = result["split_group"].map(split)
    return result

In [5]:
def set_seed(frame, seed):
    result = frame.copy()
    def split(group):
        bucket = int(hashlib.sha256(f"{seed}:{group}".encode()).hexdigest()[:8], 16) % 100
        return "train" if bucket < 70 else "validation" if bucket < 85 else "test"
    result["split"] = result.split_group.map(split)
    return result

## Первый пилот

Оставляли проверенные объявления с оплатой за смену на руки, городом, одной профессией и известным графиком без вахты. Для вилок верхняя граница могла превышать нижнюю максимум вдвое. Затем оставляли сегменты «город × профессия» с достаточным числом обучающих объявлений и работодателей.

In [6]:
PILOT_VERSION = "salary_start_v1"


BASE_FEATURES = ["city", "profession", "График", "Занятость", "Опыт работы",
            "Количество рабочих часов в день", "Количество рабочих дней в неделю",
            "Формат работы"]


CLASS_NAMES = ["below_market", "in_market", "above_market"]


@dataclass(frozen=True)
class Policy:
    min_train: int = 20
    min_reference: int = 15
    max_range_ratio: float = 2.0
    seed: int = 42

    def __post_init__(self):
        if self.min_train < 4 or self.min_reference < 4 or self.max_range_ratio < 1:
            raise ValueError("Invalid support or range policy")


def label_class(values, low, high):
    return np.where(values < low, CLASS_NAMES[0],
                    np.where(values > high, CLASS_NAMES[2], CLASS_NAMES[1]))


def prepare_pilot(frame: pd.DataFrame, policy: Policy = Policy()) -> dict:
    required = {"source", "id", "salary_raw", "city", "profession", "График",
                "analysis_status", "export_eligibility"}
    if missing := required - set(frame.columns):
        raise ValueError(f"Missing columns: {sorted(missing)}")
    data = frame.fillna("").copy().reset_index(drop=True)
    if data[["source", "id"]].eq("").any().any() or data.duplicated(["source", "id"]).any():
        raise ValueError("Source + id must be nonempty and unique")
    for col in BASE_FEATURES + ["employer_name_observed", "employer_profile_url", "title", "description"]:
        if col not in data:
            data[col] = ""
        data[col] = data[col].map(clean)
    parsed = pd.DataFrame(data.salary_raw.map(parse_salary).tolist())
    data = pd.concat([data, parsed], axis=1)
    data["exclusion_reason"] = ""
    # First reason is a mutually exclusive waterfall, not a list of all defects.
    checks = [
        (~data.analysis_status.eq("validated"), "not_validated"),
        (~data.export_eligibility.eq("included"), "eligibility_review"),
        (~data.salary_parse_status.eq("parsed"), "salary_" + data.salary_parse_status),
        (~data.pay_unit.eq("shift"), "not_explicit_shift"),
        (~data.tax_basis.eq("net"), "not_explicit_net"),
        (~data.salary_form.isin(["exact", "closed_range"]), "one_sided_offer"),
        (data.range_ratio.gt(policy.max_range_ratio), "wide_range"),
        (data.city.eq(""), "missing_city"),
        (data.profession.eq(""), "missing_profession"),
        (data.profession.str.contains(";", regex=False), "multiple_professions"),
        (data["График"].eq("Вахта"), "rotation_schedule"),
        (data["График"].eq(""), "missing_schedule"),
    ]
    for mask, reason in checks:
        take = mask & data.exclusion_reason.eq("")
        data.loc[take, "exclusion_reason"] = reason.loc[take] if isinstance(reason, pd.Series) else reason
    data["hours_stratum"] = data["Количество рабочих часов в день"].replace("", "unknown")
    # Start with the agreed broad city x profession segment. Workload and
    # schedule stay features; narrower cohorts would discard almost all data.
    data["segment"] = [json.dumps(v, ensure_ascii=False) for v in data[["city", "profession"]].values.tolist()]
    strata = ["city", "profession"]
    data["cohort"] = [json.dumps(v, ensure_ascii=False) for v in data[strata].values.tolist()]
    candidate = assign_groups(data[data.exclusion_reason.eq("")].copy(), policy.seed)
    support = []
    references = []
    for cohort, group in candidate.groupby("cohort", sort=True):
        train = group[group.split.eq("train")]
        record = {"cohort": cohort, "segment": group.segment.iloc[0],
                  **{f"n_{s}": int(group.split.eq(s).sum()) for s in ["train", "validation", "test"]},
                  "n_train_groups": train.split_group.nunique()}
        quantiles = train.salary_target.quantile([.1, .25, .5, .75, .9])
        for q, value in quantiles.items():
            record[f"p{int(q * 100)}"] = value
        record["regression_ready"] = len(train) >= policy.min_train and train.split_group.nunique() >= 3
        record["holdout_ready"] = record["regression_ready"] and record["n_validation"] >= 5 and record["n_test"] >= 5
        record["train_row_deficit"] = max(0, policy.min_train - len(train))
        record["train_group_deficit"] = max(0, 3 - train.split_group.nunique())
        record["validation_row_deficit"] = max(0, 5 - record["n_validation"])
        record["test_row_deficit"] = max(0, 5 - record["n_test"])
        support.append(record)
        if not record["regression_ready"]:
            continue
        # Holdout labels see all training employers. Training labels exclude
        # the whole current employer/duplicate component from their reference.
        for (split, own_group), rows in group.groupby(["split", "split_group"]):
            ref = train[train.split_group.ne(own_group)] if split == "train" else train
            if len(ref) < policy.min_reference or ref.split_group.nunique() < 2:
                continue
            q = ref.salary_target.quantile([.25, .5, .75])
            if q.loc[.25] >= q.loc[.75]:
                continue  # Degenerate central interval cannot support 3 classes.
            for index in rows.index:
                references.append({"row_index": index, "reference_n": len(ref),
                                   "reference_p25": q.loc[.25], "reference_p50": q.loc[.5],
                                   "reference_p75": q.loc[.75]})
    support = pd.DataFrame(support)
    if support.empty:
        raise ValueError("No comparable candidates; inspect salary formats and filters")
    ready = set(support.loc[support.regression_ready, "cohort"])
    regression = candidate[candidate.cohort.isin(ready)].copy()
    if regression.empty:
        raise ValueError("No supported training cohorts; lower thresholds only with justification or collect more data")
    ref_frame = pd.DataFrame(references).set_index("row_index") if references else pd.DataFrame()
    classes = regression.join(ref_frame, how="inner") if not ref_frame.empty else regression.iloc[:0].copy()
    if classes.empty:
        raise ValueError("No nondegenerate class references; collect more comparable observations")
    classes["market_class"] = label_class(classes.salary_target, classes.reference_p25, classes.reference_p75)
    classes["offer_salary"] = classes.salary_target
    classes["below_p25_binary"] = classes.market_class.eq("below_market").astype(int)
    unsupported = candidate[~candidate.cohort.isin(ready)].index
    data.loc[unsupported, "exclusion_reason"] = "insufficient_training_support"
    data.loc[regression.index, "exclusion_reason"] = "included"
    count_index = pd.MultiIndex.from_product([sorted(classes.cohort.unique()),
        ["train", "validation", "test"], CLASS_NAMES], names=["cohort", "split", "market_class"])
    class_counts = classes.groupby(["cohort", "split", "market_class"]).size().reindex(count_index, fill_value=0).rename("n").reset_index()
    # Compact audit: preserve original salary, bounds and every disposition.
    audit_cols = ["source", "id", "salary_raw", "city", "profession", "График",
                  "hours_stratum", *parsed.columns, "exclusion_reason"]
    audit = data[audit_cols].copy()
    common = ["source", "id", "source_url", "observed_at", "split", "split_group", "segment", "cohort",
              *BASE_FEATURES, "salary_raw", "salary_form", "salary_lower", "salary_upper", "salary_target",
              "pay_unit", "tax_basis", "range_ratio"]
    common = list(dict.fromkeys(c for c in common if c in regression.columns))
    regression = regression[common].sort_values(["split", "source", "id"]).reset_index(drop=True)
    classes = classes[common + ["offer_salary", "reference_n", "reference_p25", "reference_p50",
                               "reference_p75", "market_class", "below_p25_binary"]].sort_values(["split", "source", "id"]).reset_index(drop=True)
    summary = {"version": PILOT_VERSION, "policy": asdict(policy), "input_rows": len(frame),
               "regression_rows": len(regression), "three_class_rows": len(classes),
               "regression_splits": regression.split.value_counts().to_dict(),
               "classification_splits": classes.split.value_counts().to_dict(),
               "class_counts": pd.crosstab(classes.split, classes.market_class).to_dict(),
               "exclusions": audit.exclusion_reason.value_counts().to_dict(),
               "features_regression": BASE_FEATURES, "features_classification": BASE_FEATURES + ["offer_salary"],
               "quantiles_required": [.25, .5, .75], "quantiles_optional": [.1, .9],
               "warning": "Published shift/net offers; midpoint proxy; experimental market definition; employer holdout, not temporal validation."}
    return dict(regression=regression, three_class=classes, audit=audit,
                segment_support=support, class_support=class_counts, summary=summary)

In [7]:
pilot = prepare_pilot(raw)
pilot_counts = pilot["audit"].exclusion_reason.value_counts()
display(pd.Series({
    "Исходных строк": len(raw),
    "Кандидатов после фильтров": int(pilot_counts["included"] + pilot_counts.get("insufficient_training_support", 0)),
    "В эксперименте": len(pilot["regression"]),
    "Без распознанной оплаты за смену после предыдущих проверок": pilot_counts.get("not_explicit_shift", 0),
}).to_frame("Объявлений"))
display(pilot["regression"].split.value_counts().reindex(["train", "validation", "test"]).to_frame("Объявлений"))

,Объявлений
Исходных строк,17691
Кандидатов после фильтров,782
В эксперименте,643
Без распознанной оплаты за смену после предыдущих проверок,12188


,Объявлений
split,
train,462
validation,75
test,106


In [8]:
def fit_pilot_model(frame, *, alpha=None, binary=False):
    columns = list(BASE_FEATURES)
    transformers = [("categorical", OneHotEncoder(handle_unknown="ignore"), list(BASE_FEATURES))]
    if alpha is None:
        columns += ["offer_salary"]
        transformers.append(("offer", "passthrough", ["offer_salary"]))
        estimator = RandomForestClassifier(n_estimators=100, max_depth=6,
                                          min_samples_leaf=5, class_weight="balanced",
                                          random_state=42, n_jobs=1)
        target = "below_p25_binary" if binary else "market_class"
    else:
        estimator = GradientBoostingRegressor(loss="quantile", alpha=alpha,
                                             n_estimators=60, max_depth=2,
                                             min_samples_leaf=10, random_state=42)
        target = "salary_target"
    model = make_pipeline(ColumnTransformer(transformers), estimator)
    train = frame[frame.split.eq("train")]
    if train.empty:
        raise ValueError("Empty training data")
    model.fit(train[columns], train[target])
    return model, columns


def class_metrics(y, pred, labels):
    counts = pd.Series(y).value_counts()
    return {"n": len(y), "accuracy": accuracy_score(y, pred),
            "macro_f1": f1_score(y, pred, labels=labels, average="macro", zero_division=0),
            # Fixed-label macro recall; equals balanced accuracy if all classes occur.
            "balanced_accuracy_fixed_labels": recall_score(y, pred, labels=labels, average="macro", zero_division=0),
            "all_classes_present": all(counts.get(c, 0) > 0 for c in labels),
            "labels": list(labels), "confusion_matrix": confusion_matrix(y, pred, labels=labels).tolist()}


def run_pilot(regression, classes, *, optional=False):
    """Fit on train only. Hyperparameters are fixed before any holdout scoring."""
    for frame in [regression, classes]:
        if frame.empty or set(frame.split.unique()) != {"train", "validation", "test"}:
            raise ValueError("All three nonempty splits are required for the smoke experiments")
        if frame.groupby("split_group").split.nunique().gt(1).any():
            raise ValueError("Employer/duplicate group crosses splits")
    quantiles = [.25, .5, .75] + ([.1, .9] if optional else [])
    models = {q: fit_pilot_model(regression, alpha=q) for q in quantiles}
    classifier, class_features = fit_pilot_model(classes)
    majority = classes[classes.split.eq("train")].market_class.mode().iloc[0]
    train = regression[regression.split.eq("train")]
    reference = train.groupby("cohort").salary_target.quantile(quantiles).unstack()
    metrics = {"mandatory": {}, "optional": {}, "parameters": {
        "seed": 42, "regressor": "GradientBoostingRegressor(quantile, 60 trees, depth 2, leaf 10)",
        "classifier": "RandomForestClassifier(100 trees, depth 6, leaf 5, balanced)",
        "quantile_intervals": "raw predictions; no silent crossing correction",
        "versions": {"python": platform.python_version(), "numpy": np.__version__,
                     "pandas": pd.__version__, "sklearn": sklearn.__version__}}}
    predictions = []
    for split in ["validation", "test"]:
        holdout = regression[regression.split.eq(split)].copy()
        y = holdout.salary_target
        raw = {q: model.predict(holdout[features]) for q, (model, features) in models.items()}
        base = {q: holdout.cohort.map(reference[q]).to_numpy() for q in quantiles}
        split_result = {}
        for name, estimates in [("quantile_model", raw), ("train_segment_quantiles", base)]:
            split_result[name] = {
                "n": len(holdout), "p50_mae": mean_absolute_error(y, estimates[.5]),
                "pinball": {str(q): mean_pinball_loss(y, estimates[q], alpha=q) for q in [.25, .5, .75]},
                "p25_p75_coverage": float(np.mean((y >= estimates[.25]) & (y <= estimates[.75]))),
                "p25_p75_mean_signed_width": float(np.mean(estimates[.75] - estimates[.25])),
                "quantile_crossing_rate": float(np.mean((estimates[.25] > estimates[.5]) | (estimates[.5] > estimates[.75]))),
            }
        # Report segment sample size and model errors, not just a pooled score.
        by_segment = holdout[["cohort"]].copy()
        by_segment["abs_error"] = np.abs(y.to_numpy() - raw[.5])
        by_segment["covered"] = ((y >= raw[.25]) & (y <= raw[.75])).astype(float).to_numpy()
        segment_scores = by_segment.groupby("cohort").agg(n=("abs_error", "size"), mae=("abs_error", "mean"), coverage=("covered", "mean")).reset_index()
        segment_scores["insufficient_holdout_support"] = segment_scores.n.lt(5)
        split_result["by_segment"] = segment_scores.to_dict("records")
        ch = classes[classes.split.eq(split)]
        cp = classifier.predict(ch[class_features])
        split_result["three_class"] = {
            "learned_model": class_metrics(ch.market_class, cp, CLASS_NAMES),
            "train_majority": class_metrics(ch.market_class, np.repeat(majority, len(ch)), CLASS_NAMES)}
        rule_low = ch.cohort.map(reference[.25]).to_numpy()
        rule_high = ch.cohort.map(reference[.75]).to_numpy()
        split_result["three_class"]["reference_rule_sanity_only"] = class_metrics(
            ch.market_class, label_class(ch.offer_salary, rule_low, rule_high), CLASS_NAMES)
        split_result["three_class"]["reference_rule_note"] = (
            "Perfect by construction: validates pseudo-label reproduction, not real market accuracy.")
        quantile_prediction = holdout[["source", "id"]].copy()
        quantile_prediction["model_p25"] = raw[.25]
        quantile_prediction["model_p75"] = raw[.75]
        comparable = ch.merge(quantile_prediction, on=["source", "id"], validate="one_to_one")
        valid = comparable.model_p25.le(comparable.model_p75)
        if valid.any():
            rule_metrics = class_metrics(comparable.loc[valid, "market_class"],
                label_class(comparable.loc[valid, "offer_salary"], comparable.loc[valid, "model_p25"],
                            comparable.loc[valid, "model_p75"]), CLASS_NAMES)
            rule_metrics["n_abstained_crossing"] = int((~valid).sum())
            split_result["three_class"]["predicted_quantile_rule"] = rule_metrics
        metrics["mandatory"][split] = split_result
        prediction = holdout[["source", "id", "split", "cohort", "salary_target"]].copy()
        for q in [.25, .5, .75]:
            prediction[f"pred_p{int(q*100)}"] = raw[q]
            prediction[f"baseline_p{int(q*100)}"] = base[q]
        prediction = prediction.merge(pd.DataFrame({"source": ch.source, "id": ch.id,
                                                      "true_class": ch.market_class, "pred_class": cp}),
                                       on=["source", "id"], how="left", validate="one_to_one")
        predictions.append(prediction)
        if optional:
            metrics["optional"][split] = {
                "extended_quantiles": {str(q): mean_pinball_loss(y, raw[q], alpha=q) for q in [.1, .9]},
                "p10_p90_coverage": float(np.mean((y >= raw[.1]) & (y <= raw[.9]))),
                "p10_p90_crossing_rate": float(np.mean(raw[.1] > raw[.9]))}
    if optional:
        binary_model, binary_features = fit_pilot_model(classes, binary=True)
        for split in ["validation", "test"]:
            ch = classes[classes.split.eq(split)]
            metrics["optional"][split]["below_p25_binary"] = class_metrics(
                ch.below_p25_binary, binary_model.predict(ch[binary_features]), [0, 1])
    return metrics, pd.concat(predictions, ignore_index=True)

In [9]:
pilot_metrics, _ = run_pilot(pilot["regression"], pilot["three_class"], optional=True)
display(pd.DataFrame({
    split: {name: scores["p50_mae"] for name, scores in values.items()
            if name in ("quantile_model", "train_segment_quantiles")}
    for split, values in pilot_metrics["mandatory"].items()
}).round(1))

,validation,test
quantile_model,2580.4,1494.0
train_segment_quantiles,2598.1,1599.1


## Расширенная выборка

Сохраняем положительные числовые предложения, прошедшие проверку выгрузки. Пропуски города, профессии и условий оставляем отдельными категориями. Для недельной и дневной оплаты добавляем явное распознавание периода. Зарплату, её границы, название работодателя и свободный текст из признаков квантильной модели исключаем.

In [10]:
FEATURES = BASE_FEATURES + ["tax_basis", "Сфера деятельности компании", "Тип предприятия",
    "Тип работ", "Ассортимент", "Способ доставки", "Автомобиль"]


ALPHAS = [.25, .5, .75]


LABELS = ["below_reference", "within_reference", "above_reference"]


VARIANTS = ["narrow_raw", "bounded_raw", "all_raw", "all_log"]


MIN_TRAIN, MIN_HOLDOUT = 30, 5


def prepare(frame, seed=42):
    required = {"source", "id", "salary_raw", "analysis_status", "export_eligibility"}
    if missing := required - set(frame):
        raise ValueError(f"Missing columns: {sorted(missing)}")
    data = frame.fillna("").copy().reset_index(drop=True)
    if data[["source", "id"]].eq("").any().any() or data.duplicated(["source", "id"]).any():
        raise ValueError("Nonempty unique source/id required")
    for col in set(FEATURES + ["employer_name_observed", "employer_profile_url", "title", "description"])-{"tax_basis"}:
        if col not in data:
            data[col] = ""
        data[col] = data[col].map(clean)
    # Do not equate all anonymous users with one employer. Other name-based
    # groups remain conservative approximations (e.g. common personal names).
    grouped = data.copy()
    generic = grouped.employer_name_observed.str.casefold().isin(
        ["пользователь", "работодатель", "не указано", "неизвестно"])
    grouped.loc[generic, "employer_name_observed"] = ""
    grouped = assign_groups(grouped, seed)
    data[["split_group", "split"]] = grouped[["split_group", "split"]]
    parsed = pd.DataFrame(data.salary_raw.map(parse_offer).tolist())
    data = pd.concat([data, parsed], axis=1)
    data["disposition"] = "included_numeric"
    data.loc[~data.analysis_status.eq("validated") | ~data.export_eligibility.eq("included"), "disposition"] = "review_required"
    bad = ~data.salary_parse_status.eq("parsed") & data.disposition.eq("included_numeric")
    data.loc[bad, "disposition"] = "salary_" + data.loc[bad, "salary_parse_status"]
    data["target_kind"] = data.salary_form.map({"exact": "point", "closed_range": "point",
        "lower_only": "lower_endpoint", "upper_only": "upper_endpoint"}).fillna("")
    data["target_value"] = data.salary_target
    data.loc[data.target_kind.eq("lower_endpoint"), "target_value"] = data.salary_lower
    data.loc[data.target_kind.eq("upper_endpoint"), "target_value"] = data.salary_upper
    data["task"] = data.target_kind + "__" + data.pay_unit
    data["known_segment"] = data.city.ne("") & data.profession.ne("") & ~data.profession.str.contains(";", regex=False)
    data["wide_range"] = data.range_ratio.gt(2)
    # Freeze all source rows BEFORE any policy or target filtering.
    offers = data[data.disposition.eq("included_numeric")].copy()
    if not np.isfinite(offers.target_value).all() or not offers.target_value.gt(0).all():
        raise ValueError("Positive finite observed numeric targets required")
    # Strict whitelist. Raw text, employer identities, salary form/bounds/ratio,
    # and the structured amount field 'Оклад' are never model features.
    for col in FEATURES:
        offers[col] = offers[col].map(clean).replace("", "__MISSING__")
        unsafe = offers[col].str.contains(r"₽|\bруб\w*|зарплат|оклад|\d{4,}", case=False, regex=True)
        offers.loc[unsafe, col] = "__REDACTED_AMOUNT__"
    audit_cols = ["source", "id", "salary_raw", "salary_parse_status", "pay_unit", "tax_basis",
        "salary_form", "salary_lower", "salary_upper", "target_kind", "target_value", "range_ratio",
        "city", "profession", "known_segment", "split_group", "split", "task", "disposition"]
    audit = data[audit_cols]
    keep = list(dict.fromkeys(["source", "id", "source_url", "observed_at", "split_group", "split",
        *FEATURES, "salary_raw", "salary_form", "salary_lower", "salary_upper", "range_ratio",
        "known_segment", "wide_range", "target_kind", "target_value", "task", "pay_unit"]))
    offers = offers[[c for c in keep if c in offers]].sort_values(["task", "source", "id"]).reset_index(drop=True)
    return offers, audit

In [11]:
offers, audit = prepare(raw)
assert len(audit) == len(raw)
assert offers.groupby("split_group").split.nunique().max() == 1
assert not offers.duplicated(["source", "id"]).any()

display(audit.disposition.value_counts().to_frame("Строк"))
display(pd.crosstab(offers.target_kind, offers.pay_unit, margins=True))
display(offers[offers.target_kind.eq("point")].groupby(["pay_unit", "split"]).size()
        .unstack(fill_value=0).reindex(columns=["train", "validation", "test"]))

,Строк
disposition,
included_numeric,17246
salary_missing,432
review_required,8
salary_invalid_bounds,5


pay_unit,hour,piece,shift,unknown,week,All
target_kind,,,,,,
lower_endpoint,52,56,1793,3688,6,5595
point,59,123,2303,6950,29,9464
upper_endpoint,15,9,923,1236,4,2187
All,126,188,5019,11874,39,17246


split,train,validation,test
pay_unit,,,
hour,38,13,8
piece,84,19,20
shift,1675,288,340
unknown,4833,1155,962
week,15,8,6


## Квантили и статистический ориентир

Для P25, P50 и P75 обучаем отдельные GradientBoostingRegressor с квантильной функцией потерь. Сравниваем четыре варианта обучения. В узком варианте (`narrow_raw`) нужны известные город, профессия и график, оплата на руки, отсутствие вахты и отношение границ вилки до двух. В `bounded_raw` из этих условий остаётся ограничение на вилку. В `all_raw` используем все обучающие строки задачи, в `all_log` берём логарифм суммы и возвращаем прогноз на шкалу рублей. Для «от» и «до» проверяем два последних варианта.

Статистический ориентир ищет квантили в обучающих данных своей задачи. Начинаем с города, профессии и налоговой базы, затем убираем город, потом профессию. На каждом из этих уровней нужно минимум 20 объявлений и три группы работодателей. Если их мало, берём всю обучающую выборку задачи.

In [12]:
def training_mask(frame, variant):
    mask = frame.split.eq("train")
    if variant == "narrow_raw":
        mask &= frame.tax_basis.eq("net") & frame.known_segment & frame["График"].ne("Вахта") & frame["График"].ne("__MISSING__") & ~frame.wide_range
    elif variant == "bounded_raw":
        mask &= ~frame.wide_range
    elif variant not in ["all_raw", "all_log"]:
        raise ValueError(f"Unknown variant {variant}")
    return mask


def fit_quantiles(train, log_target=False):
    if len(train) < MIN_TRAIN or train.split_group.nunique() < 3:
        raise ValueError("Insufficient independent training support")
    target = np.log(train.target_value) if log_target else train.target_value
    models = {}
    for alpha in ALPHAS:
        model = make_pipeline(ColumnTransformer([
            ("categorical", OneHotEncoder(handle_unknown="ignore"), FEATURES)]),
            GradientBoostingRegressor(loss="quantile", alpha=alpha, n_estimators=80,
                max_depth=2, min_samples_leaf=20, random_state=42))
        model.fit(train[FEATURES], target)
        models[alpha] = model
    return models


def predict_quantiles(models, holdout, log_target=False):
    result = {q: model.predict(holdout[FEATURES]) for q, model in models.items()}
    if log_target:
        result = {q: np.exp(values) for q, values in result.items()}
    if not all(np.isfinite(v).all() for v in result.values()):
        raise ValueError("Nonfinite predictions")
    return result


def score(y, predictions):
    y = np.asarray(y)
    return {"n": len(y), "mae": mean_absolute_error(y, predictions[.5]),
        "median_absolute_error": float(np.median(np.abs(y - predictions[.5]))),
        "wape": float(np.abs(y-predictions[.5]).sum()/np.abs(y).sum()),
        "mean_pinball": float(np.mean([mean_pinball_loss(y, predictions[q], alpha=q) for q in ALPHAS])),
        **{f"pinball_p{int(q*100)}": mean_pinball_loss(y, predictions[q], alpha=q) for q in ALPHAS},
        "coverage": float(np.mean((y >= predictions[.25]) & (y <= predictions[.75]))),
        "mean_signed_width": float(np.mean(predictions[.75]-predictions[.25])),
        "crossing_rate": float(np.mean((predictions[.25] > predictions[.5]) | (predictions[.5] > predictions[.75])))}


def reference_predictions(train, holdout):
    """Train-only baseline: city+profession+tax, profession+tax, tax, task.

    Never cross unit/target tasks. Broad fallback is explicitly recorded.
    """
    levels = [["city", "profession", "tax_basis"], ["profession", "tax_basis"], ["tax_basis"]]
    tables = []
    for cols in levels:
        table = {}
        for key, group in train.groupby(cols, sort=True):
            key = key if isinstance(key, tuple) else (key,)
            if len(group) >= 20 and group.split_group.nunique() >= 3:
                table[key] = group.target_value.quantile(ALPHAS).to_numpy()
        tables.append((cols, table))
    global_q = train.target_value.quantile(ALPHAS).to_numpy()
    values, used = [], []
    for row in holdout.to_dict("records"):
        value, name = global_q, "task_only"
        for cols, table in tables:
            if any(row[c] == "__MISSING__" for c in cols):
                continue
            key = tuple(row[c] for c in cols)
            if key in table:
                value, name = table[key], "+".join(cols)
                break
        values.append(value)
        used.append(name)
    array = np.array(values)
    return {q: array[:, i] for i, q in enumerate(ALPHAS)}, used


def run_task(frame, variants=None):
    task = frame.task.iloc[0]
    variants = variants or (VARIANTS if frame.target_kind.iloc[0] == "point" else ["all_raw", "all_log"])
    records, prediction_tables, status = [], {}, []
    val, test = (frame[frame.split.eq(s)] for s in ["validation", "test"])
    for variant in variants:
        train = frame[training_mask(frame, variant)]
        base = {"task": task, "variant": variant, "n_train": len(train), "train_groups": train.split_group.nunique()}
        if len(train) < MIN_TRAIN or train.split_group.nunique() < 3 or len(val) < MIN_HOLDOUT or len(test) < MIN_HOLDOUT:
            status.append({**base, "status": "insufficient_support", "n_validation": len(val), "n_test": len(test)})
            continue
        log = variant == "all_log"
        models = fit_quantiles(train, log)
        status.append({**base, "status": "fitted", "n_validation": len(val), "n_test": len(test)})
        for split, holdout in [("validation", val), ("test", test)]:
            pred = predict_quantiles(models, holdout, log)
            baseline, levels = reference_predictions(train, holdout)
            for model_name, estimates in [("model", pred), ("baseline", baseline)]:
                records.append({**base, "split": split, "scope": "all", "estimator": model_name,
                    **score(holdout.target_value, estimates)})
                strict = (holdout.tax_basis.eq("net") & holdout.known_segment & ~holdout.wide_range &
                    holdout["График"].ne("Вахта") & holdout["График"].ne("__MISSING__")).to_numpy()
                if strict.any():
                    records.append({**base, "split": split, "scope": "common_strict", "estimator": model_name,
                        **score(holdout.target_value.to_numpy()[strict], {q: v[strict] for q,v in estimates.items()})})
            saved = holdout[["source", "id", "task", "split", "split_group", "target_value", "city", "profession",
                "tax_basis", "salary_form", "wide_range", "known_segment"]].copy()
            for q in ALPHAS:
                saved[f"pred_p{int(q*100)}"] = pred[q]
                saved[f"baseline_p{int(q*100)}"] = baseline[q]
            saved["baseline_level"] = levels
            prediction_tables[(variant, split)] = saved
    return records, prediction_tables, status

## Три категории

Для разметки используем P25 и P75 по предложениям той же задачи. При нехватке данных берём более широкую группу с той же налоговой базой. Для этого расчёта нужно минимум 30 обучающих объявлений, три группы работодателей и разные P25 и P75. При разметке обучающей вакансии её группу работодателя исключаем из расчёта порогов.

Сравниваем три способа получить категорию. Самый простой всегда выбирает частый класс. Random Forest прогнозирует класс по условиям и указанной сумме. Третий способ сравнивает сумму вакансии с P25 и P75, которые для неё предсказала квантильная модель. При пересечении границ этот способ пропускает вакансию. Macro-F1 измеряет совпадение с нашей разметкой.

In [13]:
def make_labels(frame):
    """Point offers only, with tax-specific train references and group exclusion.

    Fallback hierarchy is explicit. Unsupported reference != unusable regression.
    """
    rows = []
    point = frame[frame.target_kind.eq("point")]
    for task, group in point.groupby("task", sort=True):
        train = group[group.split.eq("train")]
        cache = {}
        for row in group.to_dict("records"):
            own = row["split_group"] if row["split"] == "train" else None
            city, profession, tax = (row[k] for k in ["city", "profession", "tax_basis"])
            cache_key = (own, city, profession, tax)
            if cache_key not in cache:
                ref = train[train.tax_basis.eq(tax)]
                if own:
                    ref = ref[ref.split_group.ne(own)]
                options = []
                if row["known_segment"]:
                    options.append(("city+profession+tax", ref[ref.city.eq(city) & ref.profession.eq(profession)]))
                if profession != "__MISSING__" and ";" not in profession:
                    options.append(("profession+tax", ref[ref.profession.eq(profession)]))
                options.append(("unit+tax", ref))
                answer = None
                for name, reference in options:
                    if len(reference) >= 30 and reference.split_group.nunique() >= 3:
                        q = reference.target_value.quantile(ALPHAS).to_numpy()
                        if q[0] < q[2]:
                            answer = (name, len(reference), *q)
                            break
                cache[cache_key] = answer
            answer = cache[cache_key]
            record = {"source": row["source"], "id": row["id"], "task": task, "split": row["split"],
                "split_group": row["split_group"], "reference_status": "supported" if answer else "insufficient_support"}
            if answer:
                name, n, low, median, high = answer
                label = LABELS[0] if row["target_value"] < low else LABELS[2] if row["target_value"] > high else LABELS[1]
                record.update(reference_level=name, reference_n=n, reference_p25=low, reference_p50=median,
                    reference_p75=high, reference_class=label)
            rows.append(record)
    return pd.DataFrame(rows)


def fit_classification(offers, labels, predictions, selected):
    records = []
    all_classes = offers.merge(labels, on=["source", "id", "task", "split", "split_group"], validate="one_to_one")
    for task, group in all_classes[all_classes.reference_status.eq("supported")].groupby("task", sort=True):
        train = group[group.split.eq("train")]
        if len(train) < MIN_TRAIN or train.reference_class.nunique() < 3 or task not in selected:
            continue
        group = group.copy()
        group["offer_value"] = group.target_value
        train = group[group.split.eq("train")]
        model = make_pipeline(ColumnTransformer([
            ("categorical", OneHotEncoder(handle_unknown="ignore"), FEATURES),
            ("amount", "passthrough", ["offer_value"])]),
            RandomForestClassifier(n_estimators=100, max_depth=6, min_samples_leaf=5,
                class_weight="balanced", random_state=42, n_jobs=1))
        model.fit(train[FEATURES+["offer_value"]], train.reference_class)
        majority = train.reference_class.mode().iloc[0]
        for split in ["validation", "test"]:
            holdout = group[group.split.eq(split)]
            if holdout.empty:
                continue
            for name, pred in [("random_forest", model.predict(holdout[FEATURES+["offer_value"]])),
                               ("majority", np.repeat(majority, len(holdout)))]:
                records.append({"task": task, "split": split, "model": name,
                    **class_metrics(holdout.reference_class, pred, LABELS)})
            pred = predictions[(task, selected[task], split)]
            joined = holdout.merge(pred[["source", "id", "pred_p25", "pred_p75"]], on=["source", "id"], validate="one_to_one")
            valid = joined.pred_p25.le(joined.pred_p75)
            if valid.any():
                j = joined[valid]
                rule = np.where(j.target_value < j.pred_p25, LABELS[0],
                    np.where(j.target_value > j.pred_p75, LABELS[2], LABELS[1]))
                records.append({"task": task, "split": split, "model": "predicted_quantile_rule",
                    "n_abstained": int((~valid).sum()), **class_metrics(j.reference_class, rule, LABELS)})
    return all_classes, records

## Обучение и выбор варианта

Вариант выбираем по среднему pinball loss трёх квантилей на валидации. Там же решаем, рекомендовать модель или статистический ориентир. Для двух основных задач повторяем выбранный вариант с разбиениями 17 и 73. Параметры модели те же, меняются обучающие и проверочные работодатели.

In [14]:
def select_variants(metrics):
    candidates = metrics[(metrics.split == "validation") & (metrics.scope == "all") & (metrics.estimator == "model")]
    return {task: rows.sort_values(["mean_pinball", "variant"]).iloc[0].variant
            for task, rows in candidates.groupby("task", sort=True)}


def recommend_estimators(metrics, selected):
    """Keep a matched train-only baseline when it beats the fitted model on validation."""
    result = {}
    for task, variant in selected.items():
        rows = metrics[(metrics.task == task) & (metrics.variant == variant) &
            (metrics.split == "validation") & (metrics.scope == "all")]
        # Prefer the simpler reference for numerical ties.
        best = rows.sort_values(["mean_pinball", "estimator"]).iloc[0]
        result[task] = best.estimator
    return result


def run(offers, *, robustness_seeds=(17, 73)):
    all_records, statuses, predictions = [], [], {}
    for task, group in offers.groupby("task", sort=True):
        print(f"Fitting {task}: {len(group)} rows", flush=True)
        records, pred, status = run_task(group)
        all_records.extend(records)
        statuses.extend(status)
        predictions.update({(task, variant, split): value for (variant, split), value in pred.items()})
    if not all_records:
        raise ValueError("No supported model tasks: require >=30 train rows, >=3 train groups, >=5 rows in both holdouts")
    metrics = pd.DataFrame(all_records)
    selected = select_variants(metrics)
    recommended = recommend_estimators(metrics, selected)
    labels = make_labels(offers)
    classes, classification = fit_classification(offers, labels, predictions, selected)
    robustness = []
    for seed in robustness_seeds:
        tagged = set_seed(offers, seed)
        for task in ["point__shift", "point__unknown"]:
            if task not in selected:
                continue
            records, _, _ = run_task(tagged[tagged.task.eq(task)], [selected[task]])
            robustness.extend({**r, "seed": seed} for r in records)
    robustness.extend({**r, "seed": 42} for r in all_records
        if r["task"] in ["point__shift", "point__unknown"] and r["variant"] == selected.get(r["task"]))
    saved_predictions = []
    for (task, variant, split), data in predictions.items():
        if selected.get(task) == variant:
            data = data.copy()
            data["variant"] = variant
            data["recommended_estimator"] = recommended[task]
            for q in ALPHAS:
                source = f"{'pred' if recommended[task] == 'model' else 'baseline'}_p{int(q*100)}"
                data[f"recommended_p{int(q*100)}"] = data[source]
            saved_predictions.append(data)
    return {"metrics": metrics, "task_status": pd.DataFrame(statuses), "selected": selected,
        "predictions": pd.concat(saved_predictions, ignore_index=True), "classes": classes,
        "classification": classification, "robustness": pd.DataFrame(robustness),
        "recommended": recommended}

In [15]:
result = run(offers)
metrics = result["metrics"]
chosen = metrics[metrics.variant.eq(metrics.task.map(result["selected"]))].copy()
classes = result["classes"]
main_tasks = ["point__shift", "point__unknown"]
task_names = {"point__shift": "За смену", "point__unknown": "Без периода"}

coverage_counts = pd.Series({
    "Числовые предложения": len(offers),
    "Строки в задачах, для которых обучилась модель": offers.task.isin(result["selected"]).sum(),
    "Точная сумма или середина вилки": offers.target_kind.eq("point").sum(),
    "Строки с условной разметкой": classes.reference_status.eq("supported").sum(),
    "Строки, для которых не хватило данных для разметки": classes.reference_status.ne("supported").sum(),
})
display(coverage_counts.to_frame("Строк"))
display(classes.reference_level.value_counts().to_frame("Строк"))

Fitting lower_endpoint__hour: 52 rows


Fitting lower_endpoint__piece: 56 rows


Fitting lower_endpoint__shift: 1793 rows


Fitting lower_endpoint__unknown: 3688 rows


Fitting lower_endpoint__week: 6 rows


Fitting point__hour: 59 rows


... (7 outputs left out)

Fitting upper_endpoint__unknown: 1236 rows


Fitting upper_endpoint__week: 4 rows


,Строк
Числовые предложения,17246
"Строки в задачах, для которых обучилась модель",17183
Точная сумма или середина вилки,9464
Строки с условной разметкой,9375
"Строки, для которых не хватило данных для разметки",89


,Строк
reference_level,
city+profession+tax,3802
unit+tax,3317
profession+tax,2256


## Ошибка прогноза суммы

MAE — среднее отклонение прогноза P50 от опубликованной суммы. Для неизвестного периода это рубли суммы из объявления. Для сравнения вариантов внутри задачи используем одни и те же проверочные вакансии.

In [16]:
main = chosen[chosen.task.isin(main_tasks) & chosen.scope.eq("all")]
display(main.pivot(index=["task", "split", "n"], columns="estimator", values="mae")
        .rename(index=task_names, columns={"baseline": "Ориентир", "model": "Модель"}).round(0))

shift_test = result["predictions"].query("task == 'point__shift' and split == 'test'").copy()
errors = (shift_test.target_value - shift_test.pred_p50).abs()
shift_metrics = main.query("task == 'point__shift' and split == 'test' and estimator == 'model'").iloc[0]
display(pd.Series({
    "Средняя абсолютная ошибка, ₽": errors.mean(),
    "Медианная абсолютная ошибка, ₽": errors.median(),
    "Покрытие P25–P75, %": 100 * shift_metrics.coverage,
    "Доля крупнейшей ошибки в сумме ошибок, %": 100 * errors.max() / errors.sum(),
}).round(1).to_frame("За смену, тест"))

estimator                    Ориентир   Модель
task        split      n                      
За смену    test       340    23153.0  23048.0
            validation 288     9470.0   9391.0
Без периода test       962    58582.0  46716.0
            validation 1155   70264.0  62290.0

,"За смену, тест"
"Средняя абсолютная ошибка, ₽",23048.4
"Медианная абсолютная ошибка, ₽",1142.9
"Покрытие P25–P75, %",43.8
"Доля крупнейшей ошибки в сумме ошибок, %",30.9


## Сравнение на строгом срезе

Берём одинаковые проверочные объявления с условиями узкого варианта. Так можно увидеть, что изменилось при расширении обучения, даже если число объявлений во всём наборе выросло.

In [17]:
strict = metrics[
    metrics.task.isin(main_tasks) & metrics.scope.eq("common_strict")
    & metrics.estimator.eq("model")
    & (metrics.variant.eq("narrow_raw") | metrics.variant.eq(metrics.task.map(result["selected"])))
]
display(strict.pivot(index=["task", "variant"], columns="split", values="mae")
        .reindex(columns=["validation", "test"]).rename(index=task_names).round(0))

split                   validation     test
task        variant                        
За смену    all_log         2078.0   2411.0
            narrow_raw      2150.0   2491.0
Без периода all_raw        38414.0  37167.0
            narrow_raw     38527.0  36085.0

## Проверка на других разбиениях

В каждом тесте сравниваем средний pinball loss модели и ориентира по P25, P50 и P75. Процент выигрыша равен `100 × (1 − ошибка модели / ошибка ориентира)`. Разбиения используют одну выгрузку, и их тестовые выборки могут пересекаться.

In [18]:
robust = result["robustness"]
losses = robust.query("scope == 'all' and split == 'test'").pivot(
    index=["task", "seed"], columns="estimator", values="mean_pinball"
)
gain = (100 * (1 - losses.model / losses.baseline)).rename("Выигрыш, %")
display(gain.unstack("seed").rename(index=task_names).round(1))

seed,17,42,73
task,,,
За смену,15.7,3.3,2.0
Без периода,13.1,19.9,10.9


## Качество разметки и прогноз границ

Для категорий смотрим Macro-F1 на тесте. Для «от» и «до» показываем MAE того способа, который выбрали на валидации. Ошибки разных единиц оплаты читаем отдельно.

In [19]:
classification = pd.DataFrame(result["classification"])
test_classes = classification[classification.task.isin(main_tasks) & classification.split.eq("test")]
display(test_classes.pivot(index="task", columns="model", values="macro_f1")
        .rename(index=task_names, columns={
            "majority": "Частый класс",
            "random_forest": "Random Forest",
            "predicted_quantile_rule": "Категория по прогнозным P25 и P75",
        }).round(3))

endpoints = chosen[
    chosen.task.str.contains("_endpoint__") & chosen.scope.eq("all") & chosen.split.eq("test")
    & chosen.estimator.eq(chosen.task.map(result["recommended"]))
]
display(endpoints[["task", "variant", "estimator", "n", "mae"]].round({"mae": 0}))

model,Частый класс,Категория по прогнозным P25 и P75,Random Forest
task,,,
За смену,0.189,0.771,0.604
Без периода,0.223,0.699,0.591


,task,variant,estimator,n,mae
12,lower_endpoint__hour,all_log,model,5,290.0
24,lower_endpoint__piece,all_log,model,6,99667.0
42,lower_endpoint__shift,all_log,baseline,310,4383.0
48,lower_endpoint__unknown,all_raw,model,479,58896.0
184,upper_endpoint__shift,all_log,model,219,8865.0
200,upper_endpoint__unknown,all_log,model,230,47236.0


## Что получилось

У объявлений без периода есть полезный сигнал для прогноза опубликованной суммы. На строгом срезе расширение обучения помогло сменной задаче, а для сумм без периода тестовая MAE немного выросла. В широкой сменной выборке несколько крайних сумм сильно влияют на среднюю ошибку. При разборе таких объявлений стоит проверить, за какой период работодатель указал сумму.

Код расчётов собран из `utils/prepare_salary_experiments.py`, `utils/run_salary_experiments.py` и `utils/salary_expanded.py`. Таблицы выше строятся из исходной выгрузки и результатов обучения.

## Паспорт расчёта

Признаки, параметры и метрики приведены выше. После запуска в `split_ids` доступны идентификаторы объявлений и части выборки для каждого разбиения. Код зафиксирован коммитом этого ноутбука. Модели обучаются в памяти.

In [20]:
dataset_git_commit = "fdbbaf1124dc675e7b7cd9ca6f9abc9b6f1a47d1"
split_columns = ["source", "id", "split_group", "split"]
split_ids = {"pilot": pilot["regression"][split_columns].copy()}
split_ids.update({
    seed: set_seed(offers, seed)[["task", *split_columns]]
    for seed in (17, 42, 73)
})